# Phase 2 Research Gate — one-click runner (Google Colab)

**What this does (fully automatic):** freezes 2 years of OKX market data + real historical
funding rates → runs the honest backtest (fees, slippage, funding, pessimistic fills) →
runs the bias audits → prints the reports you paste back to the assistant.

**How to use:**
1. Menu **Runtime → Run all** (or Ctrl/Cmd+F9)
2. Keep this browser tab open until it finishes (~30–50 min total; the data freeze is the slow part)
3. Scroll to the **last cell**, copy everything printed there, and paste it to the assistant

If Colab disconnects mid-download: just re-run the fetch cell — it resumes where it stopped
(already-frozen pairs are skipped automatically).

In [ ]:
# Cell 1 — fetch the repository (works on the merged main branch)
import os
if not os.path.exists('/content/ML_ANN_Paper_Bot'):
    os.system('git clone -q https://github.com/ah9mohammad-netizen/ML_ANN_Paper_Bot.git /content/ML_ANN_Paper_Bot')
else:
    os.system('git -C /content/ML_ANN_Paper_Bot pull -q')
os.chdir('/content/ML_ANN_Paper_Bot')
print('repo ready:', os.getcwd())

In [ ]:
# Cell 2 — dependencies
os.system('pip install -q -r requirements-research.txt')
print('deps ok')

In [ ]:
# Cell 3 — FREEZE DATA  (slow: 20–40 min; progress printed per pair)
# 730 days covers bear + chop + bull regimes. Edit DAYS below to shorten if needed.
DAYS = 730
rc = os.system(f'python -m research.fetch_data --days {DAYS}')
print('fetch return code:', rc)

In [ ]:
# Cell 4 — RUN THE GATE (truth backtest + bias audits; a few minutes)
rc = os.system('python -m research.run_research_gate --frozen')
print('gate return code:', rc)

In [ ]:
# Cell 5 — PRINT THE REPORTS  (copy everything below and paste it to the assistant)
print('=' * 30, 'REPORT 1 OF 2: REAL_W2_truth_run.md', '=' * 30)
print(open('research/output/REAL_W2_truth_run.md').read())
print('=' * 30, 'REPORT 2 OF 2: REAL_W5_bias_audits.md', '=' * 30)
print(open('research/output/REAL_W5_bias_audits.md').read())
print('=' * 30, 'END — paste everything above this line', '=' * 30)

In [ ]:
# Optional — download the full trade ledger + frozen dataset as a zip to keep
os.system('cd research && zip -qr /content/research_output.zip output data/frozen/manifest.json')
try:
    from google.colab import files
    files.download('/content/research_output.zip')
except Exception as e:
    print('download skipped (not in Colab):', e)